### 1. Definicion del problema de clasificacion
- Problema:
R: Se necesita predecir, dado el repositorio publico, si te aprobaran un pull request hacia el repositorio.
Se plantea en base a la necesidad de conocer la configuracion optima junto a los estandares especificados para conocer si un pull request sera efectivamente recibido de forma positiva por el equipo de desarrollo, para esto se ha recopilado un dataset proveniente de la API publica proporcionada por GitHub.

Se ha establecido un problema de clasificacion binario, en donde predeciremos si dada cierta cantidad de informacion correspondiente a un pull request este sera aceptado o no.

Data la complejidad, naturaleza y variedad en los repositorios, los pull request pueden variar en su formato, se vuelve entonces un problema predecir de forma directa si un pull request sera aceptado solo con observarlo, asi entonces con el análisis de multiples pull request en torno a repositorios relacionados se espera obtener una modelo capaz de predecir el destino de un pull request.

Se ha hecho uso de la API publica de github y se ha extraído 30000 datos correspondientes a pull request sobre 8 repositorios relacionados al entorno FrontEnd en el desarrollo de software.
En primera instancia se extrajo una gran cantidad de columnas (40+) con el objetivo de tener la informacion suficiente para el entrenamiento, luego se han renombrado y se ha hecho un segundo tipo de filtrado para obtener valores relevantes. Sin embargo la gran mayoria de los datos corresponde a valores de tipo string, se ha hecho una transformacion y filtrado adicional, transformando la totalidad de los datos a valores numericos priorizando la creacion de valores binarios para facilitar el trabajo del clasificador.

Luego, se da paso a la etapa de entrenamiento, comenzamos comparando la distribucion de nuestro dataset y se realiza un balance, al mismo tiempo, se entrenan multiples modelos utilizando una variedad de hiperparametros con el objetivo de encontrar el modelo y distribucion que favorece y entrega un resultado que se acomoda al problema planteado.



### 2. Diseño del experimento:


#### Extracción de datos

#### Explicacion
R: Los datos se dividieron de la siguiente forma:
El entrenamiento y prueba lo que se hizo fue tomar el dataset generado, el csv y particionarlo en variables de entrenamiento y prueba:
Cargar los datos:
```py
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
```

Generar datos de prueba y validacion:
```py

X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(
   X, 
   y, 
   test_size=.30,
   random_state=15, 
   stratify=y
)
```

In [ ]:
s

In [ ]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data.head(1)

### checks initial initial class distribution
data['merged'].value_counts()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [ ]:
load_dotenv()
github_key = os.getenv("GITHUB_TOKEN")

headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_key}"
}

urls: list[str] = []

urls.append("https://api.github.com/repos/nodejs/node/pulls?state=all")
urls.append("https://api.github.com/repos/angular/angular/pulls?state=all")
urls.append("https://api.github.com/repos/vuejs/core/pulls?state=all")
urls.append("https://api.github.com/repos/vercel/next.js/pulls?state=all")
urls.append("https://api.github.com/repos/facebook/react/pulls?state=all")
urls.append("https://api.github.com/repos/sveltejs/svelte/pulls?state=all")
urls.append("https://api.github.com/repos/withastro/astro/pulls?state=all")
urls.append("https://api.github.com/repos/QwikDev/qwik/pulls?state=all")

Se extraen datos con la utilización de API Github
El análisis se ha realizado utilizando 125 paginas, se recomienda bajar este numero para realizar la prueba.

In [ ]:
### numero de paginas a buscar, se recomienda baja este numero
max_pages = 125

def get_data(urls: list[str]) -> list[str]:
    all_results: list[str] = []
    for url in urls:
        page = 0
        print(f'Current url: {url}')
        while url:
            page += 1
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                print(f'Current page: {page}')
                data = response.json()
                
                all_results.extend(data)

                link_header = response.headers.get("Link", "")
                next_url = None
                for link in link_header.split(","):
                    if 'rel="next"' in link:
                        next_url = link[link.find("<")+1:link.find(">")]
                        break

                url = next_url

                # early return
                if page == max_pages:
                    url = None

                # 'don't get banned' check
                time.sleep(0.3)
                
            elif response.status_code == 202:
                print("Compiling data, try again shortly")
                break
            else:
                print(f"Error: {response.status_code}")
                break
    return all_results

In [ ]:
all_results = get_data(urls)

Se transforma la informacion obtenida

In [ ]:
df = pd.json_normalize(
    all_results, 
    record_path=None, 
    meta=None, 
    errors='ignore'
)
df.tail(10)

Se limpian columnas que en su totalidad sean null para evitar valores indeseados y se filtran elementos deseados.

In [ ]:
cleaned_df = df.dropna(axis=1, how='all')
filtered_df = cleaned_df[
   [
      "number", 
      "state", 
      "title", 
      "body", 
      "locked",
      "created_at",
      "updated_at",
      "closed_at",
      "merged_at",
      "assignees",
      "user.login",
      "labels",
      "author_association",
      "user.repos_url",
      "user.followers_url",
      "user.organizations_url",
      "user.starred_url",
      "user.type",
      "base.user.login",
      'base.repo.name',
      "base.user.followers_url",
      "base.user.starred_url",
      "base.repo.created_at",
      "base.repo.updated_at", 
      "base.repo.pushed_at",
      "base.repo.size",
      "base.repo.releases_url",
      "base.repo.stargazers_count",
      "base.repo.watchers_count",
      "base.repo.language",
      "base.repo.has_issues", 
      "base.repo.has_projects",
      "base.repo.has_downloads",
      "base.repo.has_wiki",
      "base.repo.has_pages",
      "base.repo.has_discussions",
      "base.repo.forks_count"
   ]
]

Se realiza un filtrado sobre las columnas sin url, en un comienzo se tenia planteado obtener aun mas información, pero la cantidad de peticiones necesarias incrementaba demasiado, se opta por filtrar.

In [ ]:
non_url_columns_df = filtered_df.drop(columns=[col for col in filtered_df.columns if col.endswith('_url')])

Se renombra columnas para facilitar su manejo.

In [ ]:
non_url_columns_df.columns = [
   'number',
   'state',
   'title',
   'body',
   'locked',
   'created_at',
   'updated_at',
   'closed_at',
   'merged_at',
   'assignees',
   'user_name',
   'labels',
   'author_association',
   'user_type',
   'repo_owner_name',
   'repo_name',
   'repo_created_at',
   'repo_updated_at',
   'repo_pushed_at',
   'repo_size',
   'repo_stargazer_count',
   'repo_watcher_count',
   'repo_language',
   'repo_has_issue',
   'repo_has_projects',
   'repo_has_downloads',
   'repo_has_wiki',
   'repo_has_pages',
   'repo_has_discussions',
   'repo_fork_count'
]

Se guarda la información obtenida para su futura manipulación

In [ ]:
non_url_columns_df.to_csv("../data/dataset/pull_request_data.csv", index=False)

## MANEJO/PREPARACIÓN DE DATOS

In [ ]:
import pandas as pd
import numpy as np
import ast

Se carga dataset previamente guardado.

In [ ]:
csv_path = '../data/dataset/pull_request_data.csv'
data = pd.read_csv(csv_path, index_col=False)

Observamos composición de columnas para entender con que tipo de columnas se trabajara

In [ ]:
data.columns

Se analiza tamaño de dataset, para considerar 

data.shape

Se crean clases iniciales

In [ ]:
### creates clases
## 1: MERGED
## 0: NOT MERGED
merged_at = data['merged_at']
merged = merged_at.notna().map({True: 1, False: 0})
data.insert(len(data.columns), 'merged', merged)

#### 2.1 Division del conjunto de datos:


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data.head(1)
### checks initial initial class distribution
data['merged'].value_counts()

In [ ]:
X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30,
                                                    random_state=15, stratify=y)

#### 2.2 Seleccion y ajuste:



In [ ]:
# Los parametros evaluados fueron gini y entropy, para el arbol de clasificacion.
# Esto se hizo, para encontrar los mejores hiperparametros para el modelo de clasificacion.
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [ ]:
clf.fit(X_train, y_train)

print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

#### 2.3 Manejo de clases desbalanceadas:

In [ ]:
# En este caso, nosotros queriamos generar multiples modelos para visualizar su rendimiento acorde al balaceo de datos.
# Por lo que, se escogio utilizar una de las tecnicas de balanceo de datos, para que los modelos puedan responder mejor ante otras clases.

# - Tecnicas:
# 1. Undersample.
# 2. Oversample.

csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data['merged'].value_counts()
X = data.drop(columns=['merged'])
y = data['merged']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=15, stratify=y)

In [ ]:
# Undersample.
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("Balanced class distribution (undersampling):")
print(y_resampled.value_counts())

train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)

In [ ]:
# Oversample.
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

print("Balanced class distribution (oversampling):")
print(y_resampled.value_counts())
train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)

Se transforma la informacion obtenida 

Se transforma la informacion obtenida 